Bu dosyada Iowa walmart datasinin EDA analizi yapilacaktir.

In [48]:
# Core libraries
from pathlib import Path

import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook utilities
from IPython.display import display

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Visualization configuration
sns.set_theme(style="whitegrid")

# Library versions
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Seaborn version: {sns.__version__}")

Pandas version: 2.2.3
NumPy version: 2.1.3
Seaborn version: 0.13.2


In [52]:
DATA_PATH = "iowa_walmart_merged.csv"

## 1. Project Overview & Business Context

### 1.1 Business Problem

Reliable sales data is essential for monitoring retail performance, evaluating product demand, comparing store operations, and supporting business decisions.

However, large-scale transactional datasets may contain missing information, inconsistent store attributes, reporting discrepancies, and other anomalies that can compromise analytical reliability.

These issues can distort key performance indicators (KPIs), create misleading sales trends, and reduce confidence in business reporting.

This project investigates these challenges through systematic data quality validation and exploratory analysis.

### 1.2 Project Objectives

The primary objectives of this project are to:

1. Assess dataset completeness, consistency, and structural integrity.
2. Identify and investigate anomalies in store information and reported sales quantities.
3. Evaluate whether reporting inconsistencies follow systematic or time-dependent patterns.
4. Quantify the potential impact of identified discrepancies on business metrics.
5. Examine sales performance across products, stores, geographic areas, and time periods.
6. Develop evidence-based recommendations to improve data reliability and business reporting.

### 1.3 Key Business Questions

**Data Quality**
- How complete and consistent is the dataset?
- Are store identifiers consistently associated with store attributes?
- Are reported sales volumes mathematically consistent with product quantities and bottle capacities?
- Do identified discrepancies follow specific temporal patterns?

**Business Impact**
- How much reported sales volume is potentially understated or overstated?
- How do these discrepancies affect monthly sales trends and volume-based KPIs?
- Could data quality issues lead to misleading operational conclusions?

**Sales Performance**
- How do sales revenues and quantities evolve over time?
- Which products and categories contribute most to overall sales?
- How does performance vary across stores and geographic areas?
- Are there meaningful seasonal patterns or unusual sales behaviors?

### 1.4 Analytical Approach

The analysis follows a structured workflow:

1. Data understanding and profiling
2. Data quality assessment
3. Controlled standardization and validation
4. Temporal anomaly investigation
5. Business impact assessment
6. Exploratory and statistical analysis
7. Business insights and recommendations

### 1.5 Scope and Limitations

The analysis is based on the information available within the provided dataset.

Observed data quality patterns will be distinguished from hypotheses about their underlying causes. Root cause conclusions will not be presented as confirmed without supporting evidence.

Missing values will not automatically be treated as formatting errors, and historical changes in store information will be evaluated before applying standardization rules.

Original source data will be preserved, and analytical corrections will be documented to support transparency and reproducibility.

## 2. Dataset Understanding

### 2.1 Dataset Description

The dataset contains retail sales records with information about invoices, transaction dates, stores, geographic locations, products, vendors, pricing, and sales quantities.

The supplied dataset contains approximately 3.55 million rows and 23 columns.

Before conducting any transformations, the dataset will be examined to understand its structure, data types, memory requirements, and potential analytical limitations.

The exact observation grain, including the uniqueness of invoice identifiers and product-level records, will be validated rather than assumed.

### 2.2 Initial Data Inspection

The initial inspection focuses on:

- Dataset dimensions
- Column availability and naming
- Inferred data types
- Memory consumption
- Representative records

No cleaning or standardization will be performed during this stage.

In [51]:
# Columns representing identifiers rather than quantities
identifier_columns = [
    "invoice_id",
    "store_no",
    "store_zip_code",
    "county_fips_code",
    "category_code",
    "vendor_number",
    "item_no",
]

# Preserve identifier formatting during ingestion
identifier_dtypes = {
    column: "string"
    for column in identifier_columns
}

# Load the raw dataset
df = pd.read_csv(
    DATA_PATH,
    dtype=identifier_dtypes
)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(
    f"Memory usage: "
    f"{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

Dataset loaded successfully.
Rows: 3,552,442
Columns: 23
Memory usage: 3.19 GB


In [53]:
# Define the expected dataset schema
expected_columns = [
    "invoice_id",
    "ordered_on",
    "store_no",
    "store_name",
    "store_address",
    "store_city",
    "store_zip_code",
    "county_fips_code",
    "county_name",
    "category_code",
    "category_name",
    "vendor_number",
    "vendor_name",
    "item_no",
    "im_desc",
    "pack",
    "bottle_volume_ml",
    "state_bottle_cost",
    "state_bottle_retail",
    "sales_bottles",
    "sales_dollars",
    "sales_liters",
    "sales_gallons",
]

# Identify schema differences
missing_columns = sorted(set(expected_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(expected_columns))

print(f"Expected columns: {len(expected_columns)}")
print(f"Actual columns: {df.shape[1]}")
print(f"Missing columns: {missing_columns}")
print(f"Unexpected columns: {unexpected_columns}")

# Stop the analysis if required columns are missing
assert not missing_columns, "Required columns are missing."

Expected columns: 23
Actual columns: 23
Missing columns: []
Unexpected columns: []


In [54]:
# Preview the first records
display(df.head())

# Inspect column data types
display(
    df.dtypes.rename("dtype").to_frame()
)

# Summarize data type distribution
display(
    df.dtypes.astype(str)
      .value_counts()
      .rename_axis("dtype")
      .reset_index(name="column_count")
)

,invoice_id,ordered_on,store_no,store_name,store_address,store_city,store_zip_code,county_fips_code,county_name,category_code,category_name,vendor_number,vendor_name,item_no,im_desc,pack,bottle_volume_ml,state_bottle_cost,state_bottle_retail,sales_bottles,sales_dollars,sales_liters,sales_gallons
0,S03383200007,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011100,BLENDED WHISKIES,260,DIAGEO AMERICAS,25606,SEAGRAMS 7 CROWN BL WHISKEY,12,750,NaN,NaN,12,110.52,9.00,2.38
1,S03383200008,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011100,BLENDED WHISKIES,260,DIAGEO AMERICAS,25608,SEAGRAMS 7 CROWN BL WHISKEY,6,1750,NaN,NaN,6,98.40,10.50,2.77
2,S03383200005,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011200,STRAIGHT BOURBON WHISKIES,65,JIM BEAM BRANDS,19066,JIM BEAM,12,750,NaN,NaN,12,160.92,9.00,2.38
3,S03383200006,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011200,STRAIGHT BOURBON WHISKIES,65,JIM BEAM BRANDS,19068,JIM BEAM,6,1750,NaN,NaN,6,165.30,10.50,2.77
4,S03383200003,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011300,TENNESSEE WHISKIES,85,BROWN-FORMAN CORPORATION,26826,JACK DANIELS OLD #7 BLACK LBL,12,750,NaN,NaN,12,234.12,9.00,2.38


,dtype
invoice_id,string[python]
ordered_on,object
store_no,string[python]
store_name,object
store_address,object
store_city,object
store_zip_code,string[python]
county_fips_code,string[python]
county_name,object
category_code,string[python]


,dtype,column_count
0,object,8
1,string,7
2,float64,5
3,int64,3


### 2.3 Initial Schema Assessment

The dataset was successfully loaded with 3,552,442 records and 23 columns, with an estimated DataFrame memory footprint of 3.19 GB.

The initial data type inspection identified:

- 7 string columns representing identifiers
- 8 object columns, including transaction dates and descriptive attributes
- 5 floating-point columns representing prices, revenue, and sales volumes
- 3 integer columns representing product packaging and sales quantities

Identifier columns were explicitly loaded as strings to preserve their original formatting, including potential leading zeros.

The `ordered_on` column requires further inspection and conversion to an appropriate datetime representation.

Numeric columns will retain their original precision until their distributions, validity, and analytical requirements have been assessed.

In [55]:
# Calculate memory usage for each column
memory_usage_mb = (
    df.memory_usage(deep=True)
      .drop(index="Index", errors="ignore")
      .div(1024**2)
      .sort_values(ascending=False)
)

# Display the largest memory consumers
display(
    memory_usage_mb
    .head(15)
    .rename("memory_mb")
    .to_frame()
    .round(2)
)

,memory_mb
store_name,249.65
im_desc,235.27
vendor_name,224.70
category_name,222.48
store_address,220.90
invoice_id,212.83
ordered_on,199.88
store_city,196.92
category_code,189.72
county_name,188.71


### 2.4 Transaction Date Validation

**Business Question**

What time period does the dataset cover, and is the transaction date field suitable for time-based analysis?

**Methodology**

Inspect the original date representation before applying an explicit datetime conversion.

The validation will assess date formatting, missing values, and parsing consistency. After successful conversion, the minimum and maximum transaction dates will be used to establish the dataset's temporal coverage.

Invalid or unparseable dates will be investigated rather than silently removed.

In [56]:
# Inspect original transaction date values
print("Transaction date samples:")

display(
    df["ordered_on"]
    .dropna()
    .head(10)
    .to_frame()
)

# Check missing transaction dates
missing_dates = df["ordered_on"].isna().sum()

print(f"Missing transaction dates: {missing_dates:,}")
print(f"Current data type: {df['ordered_on'].dtype}")

Transaction date samples:


,ordered_on
0,2012-01-03
1,2012-01-03
2,2012-01-03
3,2012-01-03
4,2012-01-03
5,2012-01-03
6,2012-01-03
7,2012-01-03
8,2012-01-03
9,2012-01-03


Missing transaction dates: 0
Current data type: object


In [57]:
# Assess invoice identifier uniqueness
total_records = len(df)
unique_invoices = df["invoice_id"].nunique(dropna=True)
missing_invoice_ids = df["invoice_id"].isna().sum()

print(f"Total records: {total_records:,}")
print(f"Unique invoice IDs: {unique_invoices:,}")
print(f"Missing invoice IDs: {missing_invoice_ids:,}")
print(f"Records per unique invoice: {total_records / unique_invoices:.2f}"
      if unique_invoices else "No valid invoice IDs found.")

Total records: 3,552,442
Unique invoice IDs: 1,626,728
Missing invoice IDs: 0
Records per unique invoice: 2.18


### 2.5 Preliminary Data Structure Findings

The initial inspection identified several important characteristics of the dataset.

**Memory Usage**

The dataset occupies approximately 3.19 GB in memory. String and object columns account for a substantial portion of memory consumption, particularly store names, product descriptions, vendor names, and other descriptive attributes.

Further cardinality analysis is required before selecting appropriate memory optimization strategies.

**Transaction Dates**

The `ordered_on` column was initially loaded as an object. Sample values follow the `YYYY-MM-DD` format, and no missing transaction dates were identified.

Explicit datetime conversion and validation are required before establishing the dataset's temporal coverage.

**Invoice Identifier Uniqueness**

The dataset contains 1,626,728 unique invoice identifiers across 3,552,442 records, corresponding to approximately 2.18 records per distinct invoice identifier.

This indicates that invoice identifiers are not unique at the record level.

However, repeated invoice identifiers do not necessarily represent duplicate transactions. Additional validation is required to determine the observation grain and identify potential duplicate records.

In [58]:
# Record memory usage before date conversion
memory_before = df.memory_usage(deep=True).sum() / 1024**3

# Parse transaction dates using an explicit format
parsed_dates = pd.to_datetime(
    df["ordered_on"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Identify non-null values that failed parsing
invalid_date_mask = (
    df["ordered_on"].notna() &
    parsed_dates.isna()
)

invalid_date_count = int(invalid_date_mask.sum())

print(f"Invalid transaction dates: {invalid_date_count:,}")

# Prevent unverified date conversions
if invalid_date_count > 0:
    display(
        df.loc[
            invalid_date_mask,
            ["invoice_id", "ordered_on"]
        ].head(10)
    )

    raise ValueError(
        "Invalid transaction dates detected. "
        "Review these values before proceeding."
    )

# Apply the validated conversion
df["ordered_on"] = parsed_dates

# Release temporary objects
del parsed_dates, invalid_date_mask

# Summarize temporal coverage
print(f"Date dtype: {df['ordered_on'].dtype}")
print(f"Earliest date: {df['ordered_on'].min():%Y-%m-%d}")
print(f"Latest date: {df['ordered_on'].max():%Y-%m-%d}")
print(f"Unique transaction dates: {df['ordered_on'].nunique():,}")

memory_after = df.memory_usage(deep=True).sum() / 1024**3

print(f"Memory before: {memory_before:.2f} GB")
print(f"Memory after: {memory_after:.2f} GB")

Invalid transaction dates: 0
Date dtype: datetime64[ns]
Earliest date: 2012-01-03
Latest date: 2026-09-30
Unique transaction dates: 3,663
Memory before: 3.19 GB
Memory after: 3.02 GB


### 2.6 Column Cardinality Assessment

**Business Question**

Which descriptive attributes contain repeated values and may benefit from more memory-efficient data types?

**Methodology**

Calculate the number of distinct non-null values and the uniqueness ratio for selected categorical and descriptive columns.

Compare these measures with column-level memory consumption to identify potential optimization opportunities.

Data types will not be modified until cardinality and analytical requirements have been evaluated.

In [59]:
# Select descriptive columns for cardinality assessment
descriptive_columns = [
    "store_name",
    "store_address",
    "store_city",
    "county_name",
    "category_name",
    "vendor_name",
    "im_desc",
]

# Calculate cardinality and memory usage
cardinality_summary = pd.DataFrame({
    "unique_values": df[descriptive_columns].nunique(dropna=True),
    "memory_mb": (
        df[descriptive_columns]
        .memory_usage(deep=True, index=False)
        .div(1024**2)
    )
})

# Calculate the proportion of distinct values
cardinality_summary["unique_ratio_pct"] = (
    cardinality_summary["unique_values"]
    .div(len(df))
    .mul(100)
)

# Sort by memory consumption
cardinality_summary = (
    cardinality_summary
    .sort_values("memory_mb", ascending=False)
    .round(2)
)

display(cardinality_summary)

,unique_values,memory_mb,unique_ratio_pct
store_name,61,249.65,0.00
im_desc,2024,235.28,0.06
vendor_name,134,224.70,0.00
category_name,87,222.48,0.00
store_address,65,220.90,0.00
store_city,56,196.92,0.00
county_name,48,188.71,0.00


### 2.7 Memory Optimization

Low-cardinality descriptive columns are converted to categorical data types to reduce memory consumption.

Original values are preserved, and numerical columns remain unchanged to avoid precision loss during subsequent validation.

In [60]:
# Identify low-cardinality descriptive columns
categorical_columns = [
    "store_name",
    "store_address",
    "store_city",
    "county_name",
    "category_name",
    "vendor_name",
    "im_desc",
]

# Measure memory before optimization
memory_before = df.memory_usage(deep=True).sum() / 1024**2

# Convert low-cardinality columns
for col in categorical_columns:
    df[col] = df[col].astype("category")

# Measure memory after optimization
memory_after = df.memory_usage(deep=True).sum() / 1024**2

# Display optimization results
print(f"Memory before: {memory_before:,.2f} MB")
print(f"Memory after: {memory_after:,.2f} MB")
print(f"Memory saved: {memory_before - memory_after:,.2f} MB")
print(f"Memory reduction: {(1 - memory_after / memory_before) * 100:.2f}%")

Memory before: 3,089.45 MB
Memory after: 1,581.55 MB
Memory saved: 1,507.91 MB
Memory reduction: 48.81%


### 2.8 Preliminary Record Grain Validation

Invoice identifiers are not unique across records.

This assessment examines store identifier cardinality and repeated invoice-product combinations to better understand the dataset's observation grain.

In [61]:
# Assess store identifier cardinality
print(f"Unique stores: {df['store_no'].nunique():,}")
print(f"Unique invoice IDs: {df['invoice_id'].nunique():,}")
print(f"Unique product IDs: {df['item_no'].nunique():,}")

# Examine repeated invoice-product combinations
repeated_invoice_items = df.duplicated(
    subset=["invoice_id", "item_no"],
    keep=False
).sum()

print(
    f"Records with repeated invoice-item combinations: "
    f"{repeated_invoice_items:,}"
)

Unique stores: 60
Unique invoice IDs: 1,626,728
Unique product IDs: 2,153
Records with repeated invoice-item combinations: 3,480,675


### 2.9 Record Grain Investigation

Repeated invoice-product combinations are examined using additional transaction attributes to assess the dataset's observation grain.

Repeated identifiers are not automatically classified as duplicate records.

In [62]:
# Define candidate record identifiers
key_combinations = {
    "invoice_id": [
        "invoice_id"
    ],
    "invoice_item": [
        "invoice_id", "item_no"
    ],
    "invoice_date_item": [
        "invoice_id", "ordered_on", "item_no"
    ],
    "invoice_date_store_item": [
        "invoice_id", "ordered_on", "store_no", "item_no"
    ],
}

# Compare repeated records across candidate keys
grain_results = []

for key_name, columns in key_combinations.items():

    repeated_count = df.duplicated(
        subset=columns,
        keep=False
    ).sum()

    grain_results.append({
        "key": key_name,
        "repeated_records": repeated_count,
        "repeated_pct": repeated_count / len(df) * 100
    })

grain_summary = pd.DataFrame(grain_results)

display(grain_summary.round(2))

,key,repeated_records,repeated_pct
0,invoice_id,3509600,98.79
1,invoice_item,3480675,97.98
2,invoice_date_item,3480675,97.98
3,invoice_date_store_item,3480675,97.98


In [63]:
# Identify repeated invoice-date-store-item combinations
candidate_key = [
    "invoice_id",
    "ordered_on",
    "store_no",
    "item_no"
]

repeated_mask = df.duplicated(
    subset=candidate_key,
    keep=False
)

# Inspect a small sample without copying all matching records
sample_indices = np.flatnonzero(
    repeated_mask.to_numpy()
)[:15]

sample_columns = candidate_key + [
    "sales_bottles",
    "sales_dollars",
    "sales_liters"
]

display(
    df.iloc[sample_indices][sample_columns]
    .sort_values(candidate_key)
)

,invoice_id,ordered_on,store_no,item_no,sales_bottles,sales_dollars,sales_liters
4,S03383200003,2012-01-03,3562,26826,12,234.12,9.00
5,S03383200004,2012-01-03,3562,26828,6,240.06,10.50
2,S03383200005,2012-01-03,3562,19066,12,160.92,9.00
3,S03383200006,2012-01-03,3562,19068,6,165.30,10.50
0,S03383200007,2012-01-03,3562,25606,12,110.52,9.00
1,S03383200008,2012-01-03,3562,25608,6,98.40,10.50
6,S03383200009,2012-01-03,3562,10008,6,91.98,10.50
9,S03383200010,2012-01-03,3562,41592,12,138.72,9.00
8,S03383200011,2012-01-03,3562,35918,18,186.84,31.50
14,S03383200012,2012-01-03,3562,42716,12,124.20,9.00


### 2.10 Exact Duplicate Assessment

This analysis distinguishes repeated identifiers from fully identical records.

Exact duplicates are identified across all 23 columns. No records are removed at this stage.

In [64]:
# Identify exact duplicates across all columns
exact_duplicate_count = df.duplicated().sum()

# Count all records belonging to duplicate groups
duplicate_group_records = df.duplicated(
    keep=False
).sum()

print(f"Total records: {len(df):,}")
print(f"Exact duplicate rows beyond first occurrences: {exact_duplicate_count:,}")
print(f"Records in exact duplicate groups: {duplicate_group_records:,}")

print(
    f"Exact duplicate rate: "
    f"{exact_duplicate_count / len(df) * 100:.2f}%"
)

Total records: 3,552,442
Exact duplicate rows beyond first occurrences: 1,794,397
Records in exact duplicate groups: 3,466,927
Exact duplicate rate: 50.51%


In [65]:
# Define the candidate transaction key
candidate_key = [
    "invoice_id",
    "ordered_on",
    "store_no",
    "item_no",
]

# Identify repeated key combinations
repeated_mask = df.duplicated(
    subset=candidate_key,
    keep=False
)

if repeated_mask.any():

    # Select one repeated key
    example = df.loc[
        repeated_mask, candidate_key
    ].iloc[0]

    # Retrieve all records with the same key
    matching_rows = df.loc[
        df["invoice_id"].eq(example["invoice_id"])
        & df["ordered_on"].eq(example["ordered_on"])
        & df["store_no"].eq(example["store_no"])
        & df["item_no"].eq(example["item_no"])
    ]

    print(f"Matching records: {len(matching_rows):,}")

    display(matching_rows.head(10))

else:
    print("No repeated candidate keys found.")

Matching records: 2


,invoice_id,ordered_on,store_no,store_name,store_address,store_city,store_zip_code,county_fips_code,county_name,category_code,category_name,vendor_number,vendor_name,item_no,im_desc,pack,bottle_volume_ml,state_bottle_cost,state_bottle_retail,sales_bottles,sales_dollars,sales_liters,sales_gallons
0,S03383200007,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011100,BLENDED WHISKIES,260,DIAGEO AMERICAS,25606,SEAGRAMS 7 CROWN BL WHISKEY,12,750,NaN,NaN,12,110.52,9.00,2.38
1749879,S03383200007,2012-01-03,3562,WAL-MART 0797 / W BURLINGTON,324 WEST AGENCY RD,WEST BURLINGTON,52655,19057,DES MOINES,1011100,BLENDED WHISKIES,260,DIAGEO AMERICAS,25606,SEAGRAMS 7 CROWN BL WHISKEY,12,750,NaN,NaN,12,110.52,9.00,2.38
